In [ ]:
"""Attentiveness module
--------------------------------------
This file is responsible foe detecting and analyzing user attention"""


with open ("attentiveness.py","r+")  as f:
    old_content=f.read()
    f.seek(0)
    f.write('"""\n' + description + '\n"""\n\n'+old_content)

In [6]:
import numpy as np
from datetime import datetime, UTC
import json
import logging
import os

# ---------------------------
# Logging Setup
# ---------------------------
logging.basicConfig(
    filename="system_log.txt",
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s]: %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)

# ---------------------------
# Utility Functions
# ---------------------------

def get_utc_timestamp():
    """Return a timezone-aware UTC timestamp in ISO format."""
    return datetime.now(UTC).isoformat()

def safe_euler_to_angles(euler_angles):
    """
    Convert Euler angles (numpy array) to floats safely.
    Handles ndim > 1 cases and ensures clean output.
    """
    try:
        pitch, yaw, roll = map(float, np.ravel(euler_angles)[:3])
        return pitch, yaw, roll
    except Exception as e:
        logging.error(f"Invalid Euler angles: {euler_angles}, Error: {e}")
        raise ValueError("Euler angles input must contain at least 3 numeric values.")

def save_to_json(data, filename="output.json"):
    """Save dictionary data into a JSON file with timestamp."""
    data["saved_at"] = get_utc_timestamp()
    with open(filename, "w") as f:
        json.dump(data, f, indent=4)
    logging.info(f"Data saved to {filename}")

def load_from_json(filename="output.json"):
    """Load dictionary data from a JSON file if it exists."""
    if not os.path.exists(filename):
        logging.warning(f"{filename} not found. Returning empty dict.")
        return {}
    with open(filename, "r") as f:
        return json.load(f)

# ---------------------------
# Core Program
# ---------------------------

def main():
    # Example: Euler angles from sensor/algorithm
    euler_angles = np.array([30.0, 60.0, 90.0])  # Replace with your real source

    # ✅ Fix for DeprecationWarning (line 107 originally)
    pitch, yaw, roll = safe_euler_to_angles(euler_angles)

    # ✅ Fix for DeprecationWarning (line 140 originally)
    timestamp = get_utc_timestamp()

    # Create data packet
    data = {
        "timestamp": timestamp,
        "euler_angles": {"pitch": pitch, "yaw": yaw, "roll": roll},
        "status": "OK"
    }

    # Save results to JSON file
    save_to_json(data, filename="sensor_output.json")

    # Load it back
    loaded_data = load_from_json("sensor_output.json")
    print("Loaded Data:", loaded_data)

    # Log event
    logging.info(f"Processed Euler angles: Pitch={pitch}, Yaw={yaw}, Roll={roll}")

# ---------------------------
# Run Program
# ---------------------------
if __name__ == "__main__":
    main()


Loaded Data: {'timestamp': '2025-09-29T22:10:29.990140+00:00', 'euler_angles': {'pitch': 30.0, 'yaw': 60.0, 'roll': 90.0}, 'status': 'OK', 'saved_at': '2025-09-29T22:10:29.990173+00:00'}


In [1]:
# Attention_Proxy_Notebook_fixed.py
# Run cell-by-cell in Jupyter or save as a .py and use jupytext.
# Author: For Yogesh (fixed + enhanced)
# Date: 2025-09-28

"""
Attention Proxy — End-to-end tutorial (fixed + enhanced)

Enhancements:
- Fixed datetime deprecation by using timezone-aware timestamps (datetime.now(UTC))
- Fixed NumPy scalar conversion when extracting Euler angles
- Added session logging (system_log.txt)
- Added calibration mode (press 'c') to compute baseline EAR/gaze
- Added recording toggle (press 'r') to save overlay video
- Added snapshot save (press 'p')
- Autosave CSV periodically
- Exponential smoothing for attentiveness for more stable display
- Safer mean/variance handling to avoid crashes when lists are empty
- Jupyter-friendly display mode (set USE_JUPYTER_DISPLAY True)
"""

import os
import time
import json
import math
import logging
from collections import deque
from datetime import datetime, UTC

import cv2
import numpy as np
import mediapipe as mp
import pandas as pd

# ---------- Configuration ----------
USE_JUPYTER_DISPLAY = False   # Set True if running inside Jupyter (slower, uses IPython display)
AUTOSAVE_INTERVAL_FRAMES = 300  # autosave CSV every N frames
RECORD_FPS = 20
SMOOTHING_ALPHA = 0.25  # exponential smoothing for attentiveness
CALIBRATION_SECONDS = 3
OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------- Logging ----------
logging.basicConfig(
    filename="system_log.txt",
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logging.info("Starting Attention Proxy (fixed + enhanced)")

# ---------- Landmarks / Model points ----------
LEFT_EYE = [33, 160, 158, 133, 153, 144]
RIGHT_EYE = [263, 387, 385, 362, 380, 373]
LEFT_IRIS = [468, 469, 470, 471]
RIGHT_IRIS = [473, 474, 475, 476]

MODEL_POINTS = np.array([
    (0.0, 0.0, 0.0),             # nose tip
    (0.0, -330.0, -65.0),        # chin
    (-225.0, 170.0, -135.0),     # left eye left corner
    (225.0, 170.0, -135.0),      # right eye right corner
    (-150.0, -150.0, -125.0),    # left mouth corner
    (150.0, -150.0, -125.0)      # right mouth corner
], dtype=np.float64)

# ---------- Helpers ----------
def now_utc_iso():
    return datetime.now(UTC).isoformat()

def safe_mean(seq, default=np.nan):
    arr = np.array(list(seq), dtype=float)
    if arr.size == 0:
        return default
    return float(np.nanmean(arr))

def safe_var(seq, default=0.0):
    arr = np.array(list(seq), dtype=float)
    if arr.size == 0:
        return default
    return float(np.nanvar(arr))

def ensure_dir(path):
    if not os.path.exists(path):
        os.makedirs(path, exist_ok=True)

# ---------- Vision utilities ----------
def eye_aspect_ratio(landmarks, eye_indices, image_w, image_h):
    try:
        pts = [(landmarks[idx].x * image_w, landmarks[idx].y * image_h) for idx in eye_indices]
        A = np.linalg.norm(np.array(pts[1]) - np.array(pts[5]))
        B = np.linalg.norm(np.array(pts[2]) - np.array(pts[4]))
        C = np.linalg.norm(np.array(pts[0]) - np.array(pts[3]))
        return 0.0 if C == 0 else (A + B) / (2.0 * C)
    except Exception as e:
        logging.debug(f"EAR error: {e}")
        return 0.0

def iris_center(landmarks, iris_indices, image_w, image_h):
    pts = [(landmarks[i].x * image_w, landmarks[i].y * image_h) for i in iris_indices]
    xs = [p[0] for p in pts]
    ys = [p[1] for p in pts]
    return (float(np.mean(xs)), float(np.mean(ys)))

def gaze_proxy(landmarks, image_w, image_h):
    try:
        lix, _ = iris_center(landmarks, LEFT_IRIS, image_w, image_h)
        rix, _ = iris_center(landmarks, RIGHT_IRIS, image_w, image_h)
        left_corner_x = (landmarks[33].x * image_w + landmarks[133].x * image_w) / 2
        right_corner_x = (landmarks[362].x * image_w + landmarks[263].x * image_w) / 2
        left_dx = (lix - left_corner_x) / max(1.0, abs(left_corner_x))
        right_dx = (rix - right_corner_x) / max(1.0, abs(right_corner_x))
        return float((left_dx + right_dx) / 2.0)
    except Exception as e:
        logging.debug(f"Gaze proxy error: {e}")
        return 0.0

def get_head_pose(landmarks, image_w, image_h, camera_matrix=None):
    try:
        image_pts = np.array([
            (landmarks[1].x * image_w, landmarks[1].y * image_h),      # nose tip
            (landmarks[152].x * image_w, landmarks[152].y * image_h),  # chin
            (landmarks[33].x * image_w, landmarks[33].y * image_h),    # left eye outer
            (landmarks[263].x * image_w, landmarks[263].y * image_h),  # right eye outer
            (landmarks[61].x * image_w, landmarks[61].y * image_h),    # left mouth corner
            (landmarks[291].x * image_w, landmarks[291].y * image_h)   # right mouth corner
        ], dtype=np.float64)
    except Exception as e:
        logging.debug(f"Head pose image_pts error: {e}")
        return None

    if camera_matrix is None:
        focal_length = image_w
        center = (image_w / 2, image_h / 2)
        camera_matrix = np.array([[focal_length, 0, center[0]],
                                  [0, focal_length, center[1]],
                                  [0, 0, 1]], dtype='double')
    dist_coeffs = np.zeros((4, 1))
    try:
        solve = cv2.solvePnP(MODEL_POINTS, image_pts, camera_matrix, dist_coeffs, flags=cv2.SOLVEPNP_ITERATIVE)
        # robust unpacking
        if isinstance(solve, tuple) and len(solve) >= 3:
            success = bool(solve[0])
            rotation_vector = solve[1]
            translation_vector = solve[2]
        else:
            logging.debug("Unexpected solvePnP return")
            return None
    except Exception as e:
        logging.debug(f"solvePnP error: {e}")
        return None

    if not success:
        return None
    try:
        rmat, _ = cv2.Rodrigues(rotation_vector)
        pose_mat = cv2.hconcat((rmat, translation_vector))
        _, _, _, _, _, _, euler_angles = cv2.decomposeProjectionMatrix(pose_mat)
        # euler_angles may be shape (3,1) - flatten safely
        pitch, yaw, roll = map(float, np.ravel(euler_angles)[:3])
        return {'pitch': pitch, 'yaw': yaw, 'roll': roll, 'rvec': rotation_vector, 'tvec': translation_vector}
    except Exception as e:
        logging.debug(f"decomposeProjectionMatrix error: {e}")
        return None

# ---------- Mediapipe setup ----------
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(static_image_mode=False, max_num_faces=1,
                                  refine_landmarks=True, min_detection_confidence=0.5, min_tracking_confidence=0.5)

# ---------- Parameters ----------
EAR_THRESHOLD = 0.20
EAR_CONSEC_FRAMES = 3
ROLLING_WINDOW_SECONDS = 10
FPS_ESTIMATE = 30

# ---------- State ----------
blink_counter = 0
total_blinks = 0
feature_log = []
rolling_buffer = deque(maxlen=ROLLING_WINDOW_SECONDS * FPS_ESTIMATE)

# smoothing state
smoothed_attentiveness = None

# calibration state
calibrated = False
calib_data = {'ears': [], 'gaze': []}
calibration_values = {'ear_baseline': None, 'gaze_baseline': 0.0, 'ear_threshold': EAR_THRESHOLD}

# autosave
frame_counter = 0

# recording
recording = False
out_writer = None

# capture
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("Could not open webcam")
image_h, image_w = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)), int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
logging.info(f"Camera opened: resolution {image_w}x{image_h}")

print("Starting capture — press 'q' to stop, 'r' to toggle recording, 'c' to calibrate, 'p' to save snapshot")

session_start = now_utc_iso()

# optional Jupyter display imports
if USE_JUPYTER_DISPLAY:
    try:
        from IPython.display import display, clear_output
        from PIL import Image as PILImage
    except Exception:
        USE_JUPYTER_DISPLAY = False
        logging.warning("Jupyter display requested but IPython/PIL not available; falling back to cv2.imshow")

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            logging.warning("Frame read failed; breaking loop")
            break

        frame_counter += 1
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = face_mesh.process(rgb)

        timestamp = now_utc_iso()
        features = {'timestamp': timestamp, 'detected': False}

        if results.multi_face_landmarks:
            lm = results.multi_face_landmarks[0].landmark
            features['detected'] = True

            left_ear = eye_aspect_ratio(lm, LEFT_EYE, image_w, image_h)
            right_ear = eye_aspect_ratio(lm, RIGHT_EYE, image_w, image_h)
            ear = (left_ear + right_ear) / 2.0
            features['ear'] = float(ear)

            # blink detection
            if ear < (calibration_values['ear_threshold'] if calibrated else EAR_THRESHOLD):
                blink_counter += 1
            else:
                if blink_counter >= EAR_CONSEC_FRAMES:
                    total_blinks += 1
                    features['blink_detected'] = True
                blink_counter = 0
            features['total_blinks'] = total_blinks

            # head pose
            pose = get_head_pose(lm, image_w, image_h)
            if pose:
                features['pitch'] = pose['pitch']
                features['yaw'] = pose['yaw']
                features['roll'] = pose['roll']
            else:
                features['pitch'] = features['yaw'] = features['roll'] = None

            # gaze
            gaze_dx = gaze_proxy(lm, image_w, image_h)
            features['gaze_dx'] = float(gaze_dx)

            # calibration accumulation (if calibrating)
            # (calibration is activated by pressing 'c' which starts a timed capture; see below)
            if calibrated and calibration_values['ear_baseline'] is None:
                # fallback: if user marked calibrated True but baseline missing, compute from calib_data
                pass

            # rolling buffer for window metrics
            rolling_buffer.append(features)
            window = list(rolling_buffer)
            if window:
                avg_ear = safe_mean([x.get('ear') for x in window if x.get('ear') is not None], default=np.nan)
                blink_count_window = sum(1 for x in window if x.get('blink_detected'))
                # approximate blink rate per minute
                blink_rate = (blink_count_window / max(1, len(window))) * (60 * FPS_ESTIMATE / ROLLING_WINDOW_SECONDS)
                yaw_var = safe_var([x.get('yaw') for x in window if x.get('yaw') is not None], default=0.0)
                gaze_mean = safe_mean([x.get('gaze_dx') for x in window if 'gaze_dx' in x], default=0.0)

                gaze_center_score = max(0, 1 - abs(gaze_mean) * 3)
                head_stability_score = max(0, 1 - (yaw_var / 100.0))
                eye_open_score = np.clip((avg_ear - 0.15) / 0.15, 0, 1)

                attentiveness = (0.5 * gaze_center_score + 0.3 * head_stability_score + 0.2 * eye_open_score) * 100
                fatigue_score = np.clip((1 - eye_open_score) * 100 + min(50, blink_rate), 0, 100)

                # smoothing
                if smoothed_attentiveness is None:
                    smoothed_attentiveness = attentiveness
                else:
                    smoothed_attentiveness = SMOOTHING_ALPHA * attentiveness + (1 - SMOOTHING_ALPHA) * smoothed_attentiveness

                features.update({
                    'attentiveness': float(attentiveness),
                    'attentiveness_smoothed': float(smoothed_attentiveness),
                    'fatigue': float(fatigue_score),
                    'blink_rate': float(blink_rate),
                    'gaze_mean': float(gaze_mean)
                })
            else:
                features.update({'attentiveness': None, 'fatigue': None, 'blink_rate': None, 'gaze_mean': None})

        feature_log.append(features)

        # overlay
        overlay = frame.copy()
        if features.get('detected'):
            attn_val = features.get('attentiveness_smoothed', features.get('attentiveness', 0))
            cv2.putText(overlay, f"Attn: {attn_val:.1f}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
            cv2.putText(overlay, f"Fatigue: {features.get('fatigue',0):.1f}", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
            cv2.putText(overlay, f"Blinks: {features.get('total_blinks',0)}", (10, 90), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
            # small HUD for pitch/yaw
            if features.get('pitch') is not None:
                cv2.putText(overlay, f"Yaw:{features.get('yaw'):.1f}", (10, 120), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 1)
        else:
            cv2.putText(overlay, "Face not detected", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

        # display
        if USE_JUPYTER_DISPLAY:
            try:
                from IPython.display import clear_output, display
                from PIL import Image as PILImage
                img_rgb = cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)
                clear_output(wait=True)
                display(PILImage.fromarray(img_rgb))
            except Exception as e:
                logging.debug(f"Jupyter display error: {e}")
                cv2.imshow("Attention Proxy", overlay)
        else:
            cv2.imshow("Attention Proxy", overlay)

        # handle key input (works in normal script with cv2.imshow)
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'):
            logging.info("Quit requested by user")
            break
        elif key == ord('r'):
            # toggle recording
            if not recording:
                ensure_dir(OUTPUT_DIR)
                out_path = os.path.join(OUTPUT_DIR, f"session_{datetime.now(UTC).strftime('%Y%m%dT%H%M%SZ')}.avi")
                fourcc = cv2.VideoWriter_fourcc(*'XVID')
                out_writer = cv2.VideoWriter(out_path, fourcc, RECORD_FPS, (frame.shape[1], frame.shape[0]))
                recording = True
                print(f"Recording started -> {out_path}")
                logging.info(f"Recording started: {out_path}")
            else:
                recording = False
                if out_writer:
                    out_writer.release()
                out_writer = None
                print("Recording stopped")
                logging.info("Recording stopped")
        elif key == ord('p'):
            # save snapshot
            snapshot_path = os.path.join(OUTPUT_DIR, f"snapshot_{datetime.now(UTC).strftime('%Y%m%dT%H%M%SZ')}.png")
            cv2.imwrite(snapshot_path, overlay)
            print(f"Saved snapshot: {snapshot_path}")
            logging.info(f"Saved snapshot: {snapshot_path}")
        elif key == ord('c'):
            # calibration routine: collect samples for CALIBRATION_SECONDS
            print(f"Calibration: Hold still and look at the center for {CALIBRATION_SECONDS} seconds...")
            logging.info("Calibration started")
            calib_start = time.time()
            calib_ears, calib_gazes = [], []
            while time.time() - calib_start < CALIBRATION_SECONDS:
                ret2, frame2 = cap.read()
                if not ret2:
                    break
                rgb2 = cv2.cvtColor(frame2, cv2.COLOR_BGR2RGB)
                res2 = face_mesh.process(rgb2)
                if res2.multi_face_landmarks:
                    lm2 = res2.multi_face_landmarks[0].landmark
                    earl = (eye_aspect_ratio(lm2, LEFT_EYE, image_w, image_h) + eye_aspect_ratio(lm2, RIGHT_EYE, image_w, image_h)) / 2.0
                    gazel = gaze_proxy(lm2, image_w, image_h)
                    calib_ears.append(earl)
                    calib_gazes.append(gazel)
                # small delay to avoid hammering
                time.sleep(0.02)
            if calib_ears:
                ear_baseline = float(np.mean(calib_ears))
                gaze_baseline = float(np.mean(calib_gazes)) if calib_gazes else 0.0
                # set adaptive ear threshold (a bit below baseline open level)
                ear_threshold = max(0.12, ear_baseline * 0.75)
                calibration_values.update({'ear_baseline': ear_baseline, 'gaze_baseline': gaze_baseline, 'ear_threshold': ear_threshold})
                calibrated = True
                print(f"Calibration done. ear_baseline={ear_baseline:.3f}, ear_threshold={ear_threshold:.3f}, gaze_baseline={gaze_baseline:.3f}")
                logging.info(f"Calibration result: {calibration_values}")
            else:
                print("Calibration failed: face not detected during calibration.")
                logging.warning("Calibration failed: no face detected")
        # write video frame if recording
        if recording and out_writer:
            try:
                out_writer.write(overlay)
            except Exception as e:
                logging.warning(f"Video write error: {e}")

        # autosave CSV periodically
        if frame_counter % AUTOSAVE_INTERVAL_FRAMES == 0 and frame_counter > 0:
            try:
                df_partial = pd.DataFrame(feature_log)
                partial_fn = os.path.join(OUTPUT_DIR, f"attention_partial_{datetime.now(UTC).strftime('%Y%m%dT%H%M%SZ')}.csv")
                df_partial.to_csv(partial_fn, index=False)
                logging.info(f"Autosaved partial CSV: {partial_fn}")
            except Exception as e:
                logging.warning(f"Autosave failed: {e}")

finally:
    # cleanup
    cap.release()
    if out_writer:
        out_writer.release()
    cv2.destroyAllWindows()
    face_mesh.close()
    logging.info("Capture stopped, resources released")

# ---------- Save logs to CSV + session summary ----------
ensure_dir(OUTPUT_DIR)
df = pd.DataFrame(feature_log)
csv_fn = os.path.join(OUTPUT_DIR, f"attention_log_{datetime.now(UTC).strftime('%Y%m%dT%H%M%SZ')}.csv")
df.to_csv(csv_fn, index=False)
print(f"Saved log to {csv_fn}")
logging.info(f"Saved log to {csv_fn}")

session_end = now_utc_iso()
summary = {
    "session_start": session_start,
    "session_end": session_end,
    "total_frames": len(feature_log),
    "total_blinks": total_blinks,
    "avg_attentiveness": float(df['attentiveness'].mean(skipna=True)) if 'attentiveness' in df.columns else None,
    "avg_fatigue": float(df['fatigue'].mean(skipna=True)) if 'fatigue' in df.columns else None,
    "calibrated": calibrated,
    "calibration_values": calibration_values
}
summary_fn = os.path.join(OUTPUT_DIR, "session_summary.json")
with open(summary_fn, "w") as f:
    json.dump(summary, f, indent=2)
print(f"Saved session summary to {summary_fn}")
logging.info(f"Session summary saved: {summary_fn}")



Starting capture — press 'q' to stop, 'r' to toggle recording, 'c' to calibrate, 'p' to save snapshot
Saved log to output\attention_log_20250929T222314Z.csv
Saved session summary to output\session_summary.json


In [5]:
!git add attentiveness.ipynb


In [7]:
!git commit -m "added description to attentiveness file"

[main (root-commit) 0724d06] added description to attentiveness file
 2 files changed, 635 insertions(+)
 create mode 100644 README.md
 create mode 100644 attentiveness.ipynb


In [9]:
!git push origin main


fatal: 'origin' does not appear to be a git repository
fatal: Could not read from remote repository.

Please make sure you have the correct access rights
and the repository exists.
